# Technical Report:<br> End-to-End License Plate Detection and Recognition (LPDR)

**Team 5: Pufa** <br> Hemo, Mohammad Javad, Sara, Mahdi <br> Sep 24, 2026

## 1. Stage 1: License Plate Detection (YOLOv8)

### 1.1 Dataset Distribution & Spatial Geometry
The detection stage was trained on a dataset of **5,829 labeled license plate instances**. Spatial density analysis reveals a normal distribution concentrated across the central region of the vehicle images, with bounding box dimensions adhering tightly to the elongated horizontal aspect ratio of standard Iranian vehicular plates.

<br>
<img src="./output_images/labels.jpg" alt="Dataset Labels and Spatial Distribution" width="700"/>



### 1.2 Detection Architecture & Hyperparameter Tuning
License plate localization is formulated as a single-class bounding-box regression task utilizing the **Ultralytics YOLOv8 nano (`yolov8n.pt`)** architecture. The nano model was selected to maximize detection throughput and ensure Stage 1 does not bottleneck end-to-end inference latency.

**Engineering Trials and Bottleneck Solutions:**<br>

* **Dataloader Optimization:** <br>Initial configurations encountered CPU-GPU worker contention and kernel memory warnings. Calibrating the pipeline to `batch=16` and `workers=2` resolved memory thrashing and stabilized GPU throughput.

* **Directory Split & Path Routing:** <br> Early training runs threw fatal `FileNotFoundError` exceptions due to unsplit flat directories. A dedicated splitting script physically partitioned images into an 80/20 train/validation split (`images/train`, `images/val`, `labels/train`, `labels/val`), referenced via an Ultralytics-compliant `data.yaml`.


### 1.3 Detection Training Curves & Quantitative Convergence
The model trained over 25 epochs, exhibiting rapid loss convergence across bounding box regression (`box_loss`), classification (`cls_loss`), and distribution focal loss (`dfl_loss`) on both training and validation splits without evidence of late-stage overfitting.

<br>
<img src="./output_images/results.png" alt="YOLO Training and Validation Metric Curves" width="900"/>

**Key Validation Metric Convergence:**
* **Precision ($\mathbf{P}$):** Climbed steadily to **$96.0\%$** across 25 epochs.
* **Recall ($\mathbf{R}$):** Converged at **$94.0\%$**, demonstrating consistent localization even in cluttered backgrounds.
* **Mean Average Precision ($\mathbf{mAP@0.5}$):** Reached **$98.3\%$** ($0.983$).
* **Stringent IoU Precision ($\mathbf{mAP@0.5:0.95}$):** Reached **$76.3\%$**, confirming tight bounding-box alignment with minimal background margins.


### 1.4 Precision-Recall, F1-Confidence, and Confusion Analysis
Evaluating detection trade-offs demonstrates the operational viability of the detector:

<br>
<div style="display: flex; flex-wrap: wrap; gap: 10px;">
  <img src="./output_images/BoxPR_curve.png" alt="Precision-Recall Curve" width="48%"/>
  <img src="./output_images/BoxF1_curve.png" alt="F1-Confidence Curve" width="48%"/>
</div>
<br>
<div style="display: flex; flex-wrap: wrap; gap: 10px;">
  <img src="./output_images/BoxP_curve.png" alt="Precision-Confidence Curve" width="48%"/>
  <img src="./output_images/BoxR_curve.png" alt="Recall-Confidence Curve" width="48%"/>
</div>

* **Precision-Recall Curve:** <br>The Area Under the Curve (AUC) confirms an overall $\mathbf{mAP@0.5 = 0.983}$, maintaining near-perfect precision across high recall thresholds.

* **Optimal Operating Threshold:** <br>The F1-Confidence Curve achieves a peak $\mathbf{F_1 = 0.95}$ at a confidence threshold of **$0.512$**. Deploying the detector with `conf=0.40`–`0.50` provides an ideal balance, preventing background false alarms while capturing oblique plates.

* **Confusion Matrix & True Positive Ratio:**<br> Out of 1,428 validation targets, the detector correctly localized **1,389 plates (97% true positive rate)**, with only 39 missed detections (false negatives).

<br>
<div style="display: flex; flex-wrap: wrap; gap: 10px;">
  <img src="./output_images/confusion_matrix.png" alt="Confusion Matrix" width="48%"/>
  <img src="./output_images/confusion_matrix_normalized.png" alt="Normalized Confusion Matrix" width="48%"/>
</div>


### 1.5 Qualitative Detection & Cropping Samples
The localized bounding boxes are automatically cropped, resized, and passed into the downstream recognition stage:

<br>
<img src="./output_images/image_8ffa28.png" alt="Detection and Extraction Grid" width="600"/>

The detector shows high resilience across standard private (white), taxi (yellow), and governmental transit (red) plates. Edge-case failures primarily occur on severely occluded or distant vehicles at oblique angles (as shown in sample 6), establishing the hardware resolution constraints evaluated in Section 3.


## 2. License Plate Recognition (PyTorch Costum CNNs vs. Mobilenetv2)


Two distinct architectures were evaluated for the 8-character sequence classification task (7 numeric digits and 1 Persian letter): a Custom CNN built from scratch and a pre-trained MobileNetV2 backbone.

**Trials and Errors (CNN Architecture Tuning):** <br>
Designing the Custom CNN involved a steep learning curve to balance model capacity against overfitting. 

*   **Initial Massive Design:** <br> The first iteration of the Custom CNN contained a massive 14 million parameters. This model severely overfit the training data and struggled to generalize.

*   **Drastic Reduction:** <br> The network was then aggressively compressed to an underfit model that lacked the capacity to learn the complex Persian character features.

*   **Stabilization:** <br> By re-introducing structured Batch Normalization, Layer Normalization, Dropout (0.3), and Residual Skip Connections, the model stabilized at roughly 600,000 parameters.

*   **Final Optimization:** <br> To push the full plate accuracy above the 90% threshold, the spatial dimensions before the fully connected layers were manipulated by resizing the pooling window. This optimized the feature maps passed into the 8 parallel classification heads, bringing the final parameter count to 1,505,688 and achieving a stable >90% full plate accuracy.



### Architectural Benchmarks

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }
    .dataframe tbody tr th {
        vertical-align: top;
    }
    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>Model</th>
      <th>Parameters</th>
      <th>Char Accuracy (%)</th>
      <th>Full Plate Accuracy (%)</th>
      <th>Latency (ms/plate)</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Custom CNN (Scratch)</td>
      <td>1,505,688</td>
      <td>98.05</td>
      <td>90.35</td>
      <td>2.79</td>
    </tr>
    <tr>
      <td>MobileNetV2 (Backbone)</td>
      <td>2,541,560</td>
      <td>98.46</td>
      <td>91.54</td>
      <td>2.36</td>
    </tr>
  </tbody>
</table>
</div>
<br>
<img src="./output_images/phase2_benchmarks_1x2.png" alt="Benchmarks" width="900"/>

The benchmarking data reveals that while MobileNetV2 possesses over 1 million more parameters and outperforms the custom architecture across all critical metrics, the achived gap is still so small.



### Training Stability
Both models ultimately exhibited stable training dynamics. The generalization gap stabilized near the zero baseline for both architectures, verifying that neither model memorized the training dataset. Furthermore, tracking the gradient norms proved that the Custom CNN maintained healthy gradient flow without vanishing or exploding.

<img src="./output_images/phase2_losses_1x2.png" alt="Losses" width="800"/>
<img src="./output_images/phase2_accuracies_1x2.png" alt="Accuracies" width="800"/>
<img src="./output_images/phase2_grad_gengap_1x2.png" alt="Gradient and Gen Gap" width="800"/>



## 3. Real-World Deployment Viability

**Latency vs. Accuracy Trade-off**<br>
For real-time traffic camera deployment, MobileNetV2 is the definitively superior model. It achieves a higher full-plate accuracy (91.54%) while executing faster inference (2.36 ms/plate) than the Custom CNN. The stacked bar chart confirms that despite having more parameters, MobileNetV2's depthwise-separable convolutions process the crops more efficiently than the dense mathematical operations of the Custom CNN.

<img src="./output_images/accuracy_vs_latency_tradeoff.png" alt="Latency Tradeoff" width="800"/>


<img src="./output_images/latency_breakdown_stacked_real.png" alt="Latency Breakdown" width="800"/>

**Camera Distance Limits**<br>
The resolution analysis identifies the hardware limits for physical camera deployment. The model's full-plate accuracy sharply declines below a minimum bounding box area threshold. To guarantee the 90% accuracy target in production, cameras must be mounted and calibrated to capture license plates at a minimum resolution area of approximately 12,000 px.

<img src="./output_images/resolution_vs_accuracy.png" alt="Resolution vs Accuracy" width="800"/>



## 4. Visual Failure Analysis

An analysis of the system's edge cases reveals specific structural weaknesses.

<img src="./output_images/recognition_failures_grid.png" alt="Recognition Failures" width="800"/>

*   **Positional Errors:** <br> The model disproportionately struggles with the single Persian letter slot compared to the numeric digits, as proven by the error isolation chart. Characters like the Persian "س" and "ص" exhibit the highest error rates.

*   **Root Cause Diagnosis:** <br> The errors stem from visual similarities between complex Persian characters under motion blur and perspective distortion. Because Iranian plates only contain one letter slot, the network receives significantly fewer training examples for letters than for the 7 numeric digit slots, leading to an unbalanced learning distribution.


<img src="./output_images/top5_worst_characters.png" alt="Top 5 Worst Characters" width="800"/>

<img src="./output_images/character_confusion_matrix.png" alt="Top 5 Worst Characters" width="800"/>